In [ ]:
!pip install transformers torch datasets accelerate openai faiss-cpu faiss-gpu datasets langchain langchain_community langchain_groq

In [ ]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch

dataset = load_dataset("wikisql")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

# Preprocessing
def preprocess_function(examples):
    inputs = [f"SQL generation: {q}" for q in examples["question"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")


    labels = [str(sql) for sql in examples["sql"]]
    labels_tokenized = tokenizer(labels, max_length=256, truncation=True, padding="max_length")


    model_inputs["labels"] = labels_tokenized["input_ids"]

    return model_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Training configuration
# training_args = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=8,
#     num_train_epochs=3,
#     evaluation_strategy="steps",
#     save_steps=500
# )

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    evaluation_strategy="steps",
    save_steps=500,
    fp16=True,
    weight_decay=0.01,
    adam_epsilon=1e-8,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
)

# Train model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

trainer.train()


In [6]:
print(dataset["train"].column_names)

['phase', 'question', 'table', 'sql']


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
!cp -r /content/results/checkpoint-2500 /content/drive/MyDrive/


In [9]:
tokenizer.save_pretrained("/content/results/checkpoint-final")


('/content/results/checkpoint-final/tokenizer_config.json',
 '/content/results/checkpoint-final/special_tokens_map.json',
 '/content/results/checkpoint-final/spiece.model',
 '/content/results/checkpoint-final/added_tokens.json')

In [10]:
!cp -r /content/results/checkpoint-final /content/drive/MyDrive/T5-Checkpoints/


In [11]:
model.save_pretrained("/content/results/checkpoint-finall")


In [27]:
!cp -r /content/results/checkpoint-finall /content/drive/MyDrive/T5-Checkpointsmodel/
